# 05 — Gather CIFs and Isotherms

**Purpose:** Link filtered isotherms to MOF crystal structures, retrieve CIF files from multiple databases, copy matched isotherm JSON files, and normalise units on the exported copies.

**Inputs:**
- `data/filtered_isotherms*.xlsx` — filtered isotherms from notebook 01
- `data/MOFs_from_CCDC_sorted_KNOWN*.xlsx` — known MOF–CSD matches
- `data/MOFs_from_CCDC_sorted_UNKNOWN*.xlsx` — unknown/ambiguous matches
- `data/filtered_isotherms_merged_with_CCDC_manual.xlsx` — manually reviewed merge (user-curated)
- `data/nistdb.pickle` — cached NIST ISODB
- Local databases: MOSAEC, CoRE-MOF, CCDC API

**Outputs:**
- `data/filtered_isotherms_merged_with_CCDC.csv` — automated merge
- `data/csd_identifiers_for_mosaec.csv` — CSD IDs extracted from manual merge
- `data/mosaec_cif_acquisition_status.xlsx` — MOSAEC CIF retrieval audit
- `data/mosaec_core_extended_cif_acquisition_status.xlsx` — extended with CoRE-MOF results
- `data/MOSAEC_CIFs_found/`, `data/CoRE_MOF_CIFs_found/`, `data/CCDC_CIFs_found/` — retrieved CIFs
- Isotherm JSON copies (to master sheet location)
- `data/step8_unit_normalisation_audit.csv` — unit conversion audit

**Manual steps:** Several stages require importing a manually edited file (after visual/numerical inspection). These are marked with ⚠ and explained inline.

## Step 1 — Merge Isotherms with Known MOF Metadata

Performs a left join from filtered isotherms to the KNOWN MOF list by adsorbent name. For unmatched rows, attempts synonym rescue via the UNKNOWN MOF list. Exports a colour-coded Excel file:
- Light yellow: isotherm without CIF match
- Light blue: known CIF without matching isotherm
- Light pink: unknown entry without matching isotherm

In [1]:
from pathlib import Path
import shutil, tempfile
import pandas as pd

project_dir = Path.cwd()
data_dir = project_dir / "data"

def find_excel(folder, stem):
    cands = sorted(p for p in folder.glob(f"{stem}*")
                   if p.suffix.lower() in {".xlsx",".xls",".xlsm"} and not p.name.startswith("~$"))
    if not cands: raise FileNotFoundError(f"No Excel file '{stem}*' in {folder}")
    return cands[0]

def read_xl(path):
    try: return pd.read_excel(path)
    except PermissionError:
        with tempfile.TemporaryDirectory() as t:
            tmp = Path(t)/path.name; shutil.copy2(path, tmp); return pd.read_excel(tmp)

def norm_cols(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(" ","_").str.replace("-","_")
    return df

def pick_col(df, candidates, label, table):
    for c in candidates:
        if c in df.columns: return c
    raise KeyError(f"'{label}' not found in {table}. Tried: {candidates}. Available: {list(df.columns)}")

# Load three input files
df_iso = norm_cols(read_xl(find_excel(data_dir, "filtered_isotherms")))
df_known = norm_cols(read_xl(find_excel(data_dir, "MOFs_from_CCDC_sorted_KNOWN")))
df_unknown = norm_cols(read_xl(find_excel(data_dir, "MOFs_from_CCDC_sorted_UNKNOWN")))

print(f"Isotherms: {len(df_iso)}, Known MOFs: {len(df_known)}, Unknown MOFs: {len(df_unknown)}")

Isotherms: 418, Known MOFs: 307, Unknown MOFs: 33


In [2]:
# Identify columns robustly (required and optional)
def pick_optional_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

iso_name = pick_col(df_iso, ["name","nist_name","adsorbent_name"], "name", "isotherms")
iso_hash = pick_col(df_iso, ["hashkey","hash_key"], "hashkey", "isotherms")
iso_file = pick_optional_col(df_iso, ["filename","isotherm_filename"])
iso_temp = pick_optional_col(df_iso, ["temperature_(k)","temperature","temp"])

known_name = pick_col(df_known, ["name","nist_name","adsorbent_name"], "name", "known_mofs")
known_csd = pick_col(df_known, ["csd_identifier","csd_id","identifier"], "csd_id", "known_mofs")
known_doi = pick_optional_col(df_known, ["doi","doi_link"])
known_formula = pick_optional_col(df_known, ["molecular_formula","formula"])
known_disorder = pick_optional_col(df_known, ["has_disorder","disorder"])

unknown_nist = pick_optional_col(df_unknown, ["nist_name","name"])
unknown_syn = pick_optional_col(df_unknown, ["csd_identifier","csd_id"])

# Standardise working copies
iso_work = pd.DataFrame({
    "name": df_iso[iso_name],
    "isotherm_filename": df_iso[iso_file] if iso_file else "",
    "hashkey": df_iso[iso_hash],
    "temperature": df_iso[iso_temp] if iso_temp else pd.NA,
})
iso_work["name"] = iso_work["name"].astype(str).str.strip()
iso_work["isotherm_filename"] = iso_work["isotherm_filename"].fillna("").astype(str).str.strip()

known_work = pd.DataFrame({
    "name": df_known[known_name],
    "csd_identifier": df_known[known_csd],
    "doi": df_known[known_doi] if known_doi else "",
    "molecular_formula": df_known[known_formula] if known_formula else "",
    "has_disorder": df_known[known_disorder] if known_disorder else "",
})
known_work["name"] = known_work["name"].astype(str).str.strip()

# Left join: isotherms -> known MOFs
merged = iso_work.merge(known_work, on="name", how="left")

# Synonym rescue from UNKNOWN list (if columns exist)
if unknown_nist and unknown_syn:
    unknown_work = df_unknown[[unknown_nist, unknown_syn]].copy()
    unknown_work.columns = ["nist_name", "synonym_name"]
    unknown_work["nist_name"] = unknown_work["nist_name"].astype(str).str.strip()
    syn_map = dict(zip(unknown_work["nist_name"], unknown_work["synonym_name"]))

    unmatched_mask = merged["csd_identifier"].isna()
    for idx in merged[unmatched_mask].index:
        name = merged.at[idx, "name"]
        if name in syn_map:
            syn = syn_map[name]
            match = known_work[known_work["name"].str.lower() == str(syn).lower()]
            if len(match) > 0:
                for col in ["csd_identifier", "doi", "molecular_formula", "has_disorder"]:
                    merged.at[idx, col] = match.iloc[0][col]
else:
    print("Unknown MOF synonym columns not found; skipping synonym rescue.")

print(f"Merged rows: {len(merged)}")
print(f"With CIF match: {merged['csd_identifier'].notna().sum()}")
print(f"Without CIF: {merged['csd_identifier'].isna().sum()}")

# Export
merged.to_csv(data_dir / "filtered_isotherms_merged_with_CCDC.csv", index=False)
print("Saved: data/filtered_isotherms_merged_with_CCDC.csv")

Merged rows: 420
With CIF match: 279
Without CIF: 141
Saved: data/filtered_isotherms_merged_with_CCDC.csv


## Step 2 — ⚠ Import Manual Merge File

After exporting the automated merge above, the user manually reviews and edits it in Excel, highlighting rows to exclude. This cell loads that manually curated file and extracts the valid CSD identifiers for CIF retrieval.

> **Manual step:** Open `filtered_isotherms_merged_with_CCDC.xlsx`, review matches, highlight rows to exclude, save as `filtered_isotherms_merged_with_CCDC_manual.xlsx`, then run this cell.

In [3]:
from openpyxl import load_workbook

manual_path = data_dir / "filtered_isotherms_merged_with_CCDC_manual.xlsx"
if not manual_path.exists():
    raise FileNotFoundError(f"Manual merge file not found: {manual_path.name}. Create it from the Step 1 output.")

def is_highlighted(fill):
    return fill is not None and getattr(fill, "fill_type", None) not in (None, "none")

manual_df = pd.read_excel(manual_path)
manual_df.columns = [str(c).strip() for c in manual_df.columns]

# Detect highlighted rows (= excluded by user)
wb = load_workbook(manual_path, data_only=True)
ws = wb.active
highlighted = pd.Series([
    any(is_highlighted(cell.fill) for cell in ws[r])
    for r in range(2, 2 + len(manual_df))
], dtype=bool)

unhighlighted = manual_df[~highlighted].copy()
if "row_origin" in unhighlighted.columns:
    unhighlighted = unhighlighted[unhighlighted["row_origin"].astype(str).str.strip() == "isotherm_with_cif"]

csd_ids = sorted({
    c for c in unhighlighted["csd_identifier"].astype(str).str.strip().str.upper()
    if c and c.lower() not in ("nan", "none")
})

pd.DataFrame({"csd_identifier": csd_ids}).to_csv(data_dir / "csd_identifiers_for_mosaec.csv", index=False)
unhighlighted.to_excel(data_dir / "filtered_isotherm_merged_with_CCDC_manual_unhighlighted.xlsx", index=False)

print(f"Total rows: {len(manual_df)}, Highlighted (excluded): {highlighted.sum()}")
print(f"Unique CSD identifiers for CIF retrieval: {len(csd_ids)}")

Total rows: 435, Highlighted (excluded): 22
Unique CSD identifiers for CIF retrieval: 274


## Step 3 — Retrieve CIFs from MOSAEC (Local)

Searches the local MOSAEC database (neutral and charged folders) for CIF files matching the target CSD identifiers. Matches on the first 6 characters of the filename with digit-suffix rules to avoid false positives.

In [4]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
MOSAEC_CHARGED = Path(r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\Scraped DBs\MOSAEC-DB\mosaec-db_base\mosaec-db_base\full\charged")
MOSAEC_NEUTRAL = Path(r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\Scraped DBs\MOSAEC-DB\mosaec-db_base\mosaec-db_base\full\neutral")

def norm_code(v): return str(v).strip().upper()

def csd_match(csd_id, cif_stem):
    c, s = norm_code(csd_id), norm_code(cif_stem)
    if len(c)<6 or len(s)<6 or s[:6]!=c[:6]: return False
    if not any(ch.isdigit() for ch in c) and len(s)>=7 and s[6].isdigit(): return False
    if len(c)>=8 and c[6].isdigit() and c[7].isdigit():
        if len(s)<8 or s[6]!=c[6] or s[7]!=c[7]: return False
    return True

# Build candidate table
cif_sources = [("full_charged", MOSAEC_CHARGED), ("full_neutral", MOSAEC_NEUTRAL)]
candidates = []
for label, folder in cif_sources:
    if folder.exists():
        for p in folder.rglob("*.cif"):
            candidates.append({"label": label, "name": p.name, "stem": p.stem, "path": str(p)})
print(f"MOSAEC CIF candidates: {len(candidates)}")

# Match
out_dir = data_dir / "MOSAEC_CIFs_found"; out_dir.mkdir(exist_ok=True)
results = []
for cid in csd_ids:
    found = False
    for cand in candidates:
        if csd_match(cid, cand["stem"]):
            shutil.copy2(cand["path"], out_dir / cand["name"])
            results.append({"csd_identifier": cid, "cif_filename": cand["name"],
                           "cif_source": cand["label"], "database": "MOSAEC"})
            found = True; break
    if not found:
        results.append({"csd_identifier": cid, "cif_filename": "", "cif_source": "", "database": ""})

res_df = pd.DataFrame(results)
res_df.to_excel(data_dir / "mosaec_cif_acquisition_status.xlsx", index=False)
found_n = len(res_df[res_df["cif_filename"] != ""])
print(f"MOSAEC: found {found_n}/{len(csd_ids)} CIFs")

MOSAEC CIF candidates: 91478
MOSAEC: found 119/274 CIFs


# Step 3b - Editing CIFs for consistency

If the first line of CIF file contains a '#' as its first character, clear the line but don't remove it. Check through the obtained CIFs again and check if the first line of each CIF is blank

In [5]:
# Clean first lines of MOSAEC CIFs found in Step 3
cif_dir = data_dir / "MOSAEC_CIFs_found"

if not cif_dir.exists():
    print(f"⚠ Directory not found: {cif_dir}")
else:
    cif_files = sorted(cif_dir.glob("*.cif"))
    print(f"Processing MOSAEC CIFs: {len(cif_files)} files")
    
    modified_count = 0
    verification_issues = []
    
    for cif_path in cif_files:
        # Read the file
        with open(cif_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
        
        # Check and fix first line if it contains '#'
        if lines and lines[0].strip().startswith("#"):
            lines[0] = "\n"  # Clear the line but keep it
            modified_count += 1
            
            # Write back
            with open(cif_path, "w", encoding="utf-8") as f:
                f.writelines(lines)
        
        # Verify first line is blank
        if lines:
            if lines[0].strip() != "":
                verification_issues.append(f"{cif_path.name}: First line is not blank: {repr(lines[0][:50])}")
    
    print(f"\n{'='*70}")
    print(f"Total MOSAEC CIF files processed: {len(cif_files)}")
    print(f"Files with '#' in first line (modified): {modified_count}")
    if verification_issues:
        print(f"\n⚠ Verification Issues ({len(verification_issues)}):")
        for issue in verification_issues[:10]:  # Show first 10 issues
            print(f"  - {issue}")
        if len(verification_issues) > 10:
            print(f"  ... and {len(verification_issues) - 10} more")
    else:
        print(f"✓ All {len(cif_files)} MOSAEC CIF files verified: first line is blank")


Processing MOSAEC CIFs: 119 files

Total MOSAEC CIF files processed: 119
Files with '#' in first line (modified): 119
✓ All 119 MOSAEC CIF files verified: first line is blank


## Step 4 — Extend with CoRE-MOF Database

For CSD identifiers not found in MOSAEC, searches the local CoRE-MOF database (solvent-removed 2019 version). Updates the acquisition status table in-place.

In [8]:
CORE_MOF_DIR = Path(r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\Scraped DBs\CoRE MOF - DB\All_solvent_removed_2019")

mosaec_df = pd.read_excel(data_dir / "mosaec_cif_acquisition_status.xlsx")
for col in ["csd_identifier","cif_source","cif_filename","database"]:
    mosaec_df[col] = mosaec_df[col].fillna("").astype(str).str.strip()

missing_ids = sorted(set(mosaec_df.loc[mosaec_df["cif_filename"]=="", "csd_identifier"]))
core_out = data_dir / "CoRE_MOF_CIFs_found"; core_out.mkdir(exist_ok=True)

if CORE_MOF_DIR.exists() and missing_ids:
    core_cifs = list(CORE_MOF_DIR.rglob("*.cif"))
    for cid in missing_ids:
        for p in core_cifs:
            prefix = p.stem.split("_")[0].split("-")[0].split(" ")[0].upper()
            if csd_match(cid, prefix):
                shutil.copy2(p, core_out / p.name)
                mask = mosaec_df["csd_identifier"] == cid
                mosaec_df.loc[mask, "cif_filename"] = p.name
                mosaec_df.loc[mask, "cif_source"] = "CoRE-MOF"
                mosaec_df.loc[mask, "database"] = "CoRE-MOF"
                break

mosaec_df.to_excel(data_dir / "mosaec_core_extended_cif_acquisition_status.xlsx", index=False)
still_missing = len(mosaec_df[mosaec_df["cif_filename"]==""])
print(f"After CoRE-MOF: {len(csd_ids)-still_missing}/{len(csd_ids)} resolved, {still_missing} remaining")#

After CoRE-MOF: 198/274 resolved, 76 remaining


## Step 5 — Retrieve Remaining CIFs from CCDC API

Final fallback: uses the CSD Python API to retrieve crystal structures for any CSD identifiers still missing a CIF file. Requires the CSD Python kernel.

In [9]:
import ccdc.io

combined = pd.read_excel(data_dir / "mosaec_core_extended_cif_acquisition_status.xlsx")
for col in ["csd_identifier","cif_source","cif_filename","database"]:
    combined[col] = combined[col].fillna("").astype(str).str.strip()

ccdc_ids = sorted(set(combined.loc[combined["cif_filename"]=="", "csd_identifier"]))
ccdc_out = data_dir / "CCDC_CIFs_found"; ccdc_out.mkdir(exist_ok=True)

reader = ccdc.io.EntryReader("CSD")
ccdc_found, ccdc_missing = 0, 0
for cid in ccdc_ids:
    try:
        entry = reader.entry(cid)
        if entry and entry.crystal:
            fname = f"CCDC__{cid}.cif"
            with ccdc.io.CrystalWriter(str(ccdc_out / fname)) as w:
                w.write(entry.crystal)
            mask = combined["csd_identifier"] == cid
            combined.loc[mask, "cif_filename"] = fname
            combined.loc[mask, "cif_source"] = "ccdc_api"
            combined.loc[mask, "database"] = "CCDC"
            ccdc_found += 1
        else:
            ccdc_missing += 1
    except:
        ccdc_missing += 1

combined[["csd_identifier","cif_source","cif_filename","database"]].to_excel(
    data_dir / "final_cif_acquisition_status.xlsx", index=False)
print(f"CCDC API: found {ccdc_found}, not found {ccdc_missing}")
print(f"Total CIFs resolved: {len(combined[combined['cif_filename']!=''])}/{len(combined)}")

CCDC API: found 75, not found 1
Total CIFs resolved: 273/274


## Step 5b — Materials Studio Edits

Went through each of the CIFs captured by the CCDC and CoRE-MOF for issues and disorders, manually. Many were manually adjusted via Materials Studio, some were discarded.

Addiiotnally, some CIFs were created from other CIFs to fill in additional CIF gaps in the dataset.


## Step 6 — Copy Isotherm JSON Files

Copies all matched isotherm JSON files from the ISODB library to the master output folder. Uses filename-based matching. Enables clean-run mode (deletes previous output before copying).

> **Configuration:** Update `master_sheet_base`, `library_root`, and `output_dir` paths below to match your local setup.

In [10]:
import re, pickle

master_sheet_base = Path(r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF CIFs DATABASE - ALL\MOF_CIF_Isotherms_MASTER_SHEET")
library_root = Path(r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\ML Project Directory\Data Collection\isodb-library\Library")
output_dir = Path(r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF CIFs DATABASE - ALL\isotherms_full")
output_dir.mkdir(parents=True, exist_ok=True)

# Resolve master sheet path
ms_path = None
for ext in (".xlsx",".xls",".xlsm",".csv"):
    p = master_sheet_base.with_suffix(ext)
    if p.exists(): ms_path = p; break
if ms_path is None:
    ms_path = sorted(p for p in master_sheet_base.parent.glob(f"{master_sheet_base.name}*")
                     if p.suffix.lower() in (".xlsx",".xls",".xlsm",".csv") and not p.name.startswith("~$"))[0]

ms_df = pd.read_excel(ms_path) if ms_path.suffix != ".csv" else pd.read_csv(ms_path)
fn_col = next(c for c in ms_df.columns if "filename" in c.lower())
target_fns = set(ms_df[fn_col].dropna().astype(str).str.strip())

# Build filename→path index from library
fn_index = {}
for p in library_root.rglob("*.json"):
    fn_index[p.stem.lower()] = p

# Clean run: remove existing JSONs
CLEAN_RUN = True
if CLEAN_RUN:
    for p in output_dir.glob("*.json"): p.unlink()

# Copy
copied, missed = 0, 0
for fn in sorted(target_fns):
    key = fn.strip().lower()
    if key.endswith(".json"): key = key[:-5]
    if key in fn_index:
        shutil.copy2(fn_index[key], output_dir / fn_index[key].name)
        copied += 1
    else:
        missed += 1

print(f"Isotherm JSONs: copied {copied}, not found {missed}")

Isotherm JSONs: copied 398, not found 0


## Step 7 — Unit Normalisation on Exported Isotherms

Applies the same unit conversion and coherence fixes as notebook 01, but on the exported JSON copies in the master output folder. Converts adsorption to mmol/g and pressure to bar. Generates an audit CSV for traceability.

In [11]:
import json

M_CH4 = 16.043
V_STP = 22_414.0

def _ads_factor(unit):
    if unit is None: return None
    u = str(unit).strip().lower(); uns = u.replace(" ","")
    m = {'mmol/g':1.0, 'mol/kg':1.0, 'mmol/kg':1e-3, 'mg/g':1.0/M_CH4,
         'g/g':1000.0/M_CH4, 'g/l':1.0/M_CH4, 'g/ml':1000.0/M_CH4}
    if u in m: return m[u]
    if u in ('µmol/g','umol/g','μmol /g') or uns in ('µmol/g','umol/g','μmol/g'): return 1e-3
    if u in ('cm3(stp)/g','cc(stp)/g','ml(stp)/g','cm3 (stp)/g','cc (stp)/g','ml (stp)/g','ml/g'):
        return 1000.0/V_STP
    if u in ('l(stp)/g','l (stp)/g'): return 1e6/V_STP
    if u in ('wt%','wt. %','wt.%','weight %','uptake%','uptake %'): return 10.0/M_CH4
    return None

def _pres_factor(unit):
    if unit is None: return None
    u = str(unit).strip().lower()
    m = {'bar':1.0, 'mbar':1e-3, 'pa':1e-5, 'kpa':1e-2, 'mpa':10.0,
         'atm':1.01325, 'torr':1.0/750.062, 'mmhg':1.0/750.062, 'psi':0.0689476}
    return m.get(u, None)

def _norm_fn(name):
    s = str(name).strip()
    return s[:-5].lower() if s.lower().endswith(".json") else s.lower()

# Load audit removal list
audit_skip = set()
arf = data_dir / "audit_removal_filenames.txt"
if arf.exists():
    for line in arf.read_text(encoding="utf-8").splitlines():
        l = line.split("#",1)[0].strip()
        if l:
            parts = [p.strip() for p in l.split(",") if p.strip()]
            audit_skip.add(_norm_fn(parts[-1] if parts else l))

# Coherence fix filename sets
psi_fns = {_norm_fn(x) for x in ["10.1021jp304631m.Isotherm15","10.1021jp304631m.Isotherm17",
    "10.1021jp304631m.Isotherm30","10.1021jp304631m.Isotherm31",
    "10.1021jp304631m.Isotherm39","10.1021jp304631m.Isotherm40"]}
cm3_fns = {_norm_fn(x) for x in ["10.1039C3ta11548h.isotherm17","10.1039C3ta11548h.isotherm18",
    "10.1039C3ta11548h.isotherm15","10.1039C3ta11548h.isotherm16","10.1039C3ta11840a.Isotherm6"]}
div100_fns = {_norm_fn(x) for x in ["10.1039C3cc48275h.Isotherm10","10.1039C3cc48275h.Isotherm11",
    "10.1039C3cc48275h.Isotherm12","10.1039C3cc48275h.Isotherm13",
    "10.1039C3cc48275h.Isotherm14","10.1016j.micromeso.2011.09.006.isotherm5"]}

audit_rows = []
json_files = sorted(output_dir.glob("*.json"))
for jp in json_files:
    fn = _norm_fn(jp.stem)
    if fn in audit_skip:
        audit_rows.append({"filename": jp.stem, "status": "skipped_audit_list"})
        continue

    with open(jp, "r", encoding="utf-8") as f:
        iso = json.load(f)

    fixes = []
    # Coherence fixes (with idempotent guards)
    pressures = [pt.get("pressure") for pt in iso.get("isotherm_data",[]) if pt.get("pressure") is not None]
    adsorptions = [pt.get("total_adsorption") for pt in iso.get("isotherm_data",[]) if pt.get("total_adsorption") is not None]
    max_p = max(pressures) if pressures else 0
    max_q = max(adsorptions) if adsorptions else 0

    if fn in psi_fns and max_p < 0.02:
        for pt in iso["isotherm_data"]:
            if pt.get("pressure") is not None: pt["pressure"] *= 100000.0/14.5038
        fixes.append("PSI->bar")
    if fn in cm3_fns and max_q > 2.0:
        for pt in iso["isotherm_data"]:
            if pt.get("total_adsorption") is not None: pt["total_adsorption"] *= 1000.0/22414.0
            for sp in pt.get("species_data",[]):
                if sp.get("adsorption") is not None: sp["adsorption"] *= 1000.0/22414.0
        fixes.append("cm3->mmol")
    if fn in div100_fns and max_p > 40.0:
        for pt in iso["isotherm_data"]:
            if pt.get("pressure") is not None: pt["pressure"] /= 100.0
        fixes.append("P/100")

    # Unit conversion
    af = _ads_factor(iso.get("adsorptionUnits"))
    pf = _pres_factor(iso.get("pressureUnits"))
    if af and af != 1.0:
        for pt in iso["isotherm_data"]:
            if pt.get("total_adsorption") is not None: pt["total_adsorption"] *= af
            for sp in pt.get("species_data",[]):
                if sp.get("adsorption") is not None: sp["adsorption"] *= af
        iso["adsorptionUnits"] = "mmol/g"
    if pf and pf != 1.0:
        for pt in iso["isotherm_data"]:
            if pt.get("pressure") is not None: pt["pressure"] *= pf
        iso["pressureUnits"] = "bar"

    with open(jp, "w", encoding="utf-8") as f:
        json.dump(iso, f, indent=4); f.write("\n")

    audit_rows.append({
        "filename": jp.stem, "status": "processed",
        "adsorptionUnits_orig": iso.get("adsorptionUnits",""),
        "pressureUnits_orig": iso.get("pressureUnits",""),
        "ads_factor": af, "pres_factor": pf,
        "coherence_fixes": "; ".join(fixes) if fixes else "",
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(data_dir / "step8_unit_normalisation_audit.csv", index=False)
print(f"Processed {len(json_files)} JSON files")
print(f"Audit saved: data/step8_unit_normalisation_audit.csv")

Processed 398 JSON files
Audit saved: data/step8_unit_normalisation_audit.csv
